# 🤖 TelecomX — Modelagem Preditiva de Churn

---

| | |
|---|---|
| **Empresa** | TelecomX (fictícia) |
| **Projeto** | Churn de Clientes — Parte 2: Modelagem Preditiva |
| **Processo** | Pré-processamento · Modelagem · Avaliação · Interpretação |
| **Linguagem** | Python 3 |
| **Ambiente** | Google Colab |
| **Bibliotecas** | Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn · Imbalanced-learn |

---

> **Contexto:** Com os dados tratados na Parte 1, agora construímos modelos de Machine Learning
> para prever quais clientes têm maior chance de cancelar o serviço (Churn),
> permitindo ações preventivas de retenção.


---
# ⚙️ Seção 0 — Instalação e Importações

In [ ]:
# Instalar imbalanced-learn (SMOTE) caso não esteja disponível
!pip install imbalanced-learn -q


In [ ]:
import json, requests, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

# Balanceamento
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("✅ Bibliotecas importadas com sucesso!")


---
# 📂 Seção 1 — Extração e Reaproveitamento dos Dados Tratados

## 1.1 Recarregando e Retratando os Dados da API

Reutilizamos o pipeline de ETL da Parte 1 para garantir reprodutibilidade.
Os dados são carregados diretamente da API do GitHub e passam pelo mesmo tratamento anterior.


In [ ]:
API_URL = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science/main/TelecomX_Data.json"

response = requests.get(API_URL)
raw_data = response.json()
print(f"✅ {len(raw_data)} registros carregados da API.")

# ── Flatten do JSON aninhado (mesmo pipeline da Parte 1) ──────────────
records = []
for d in raw_data:
    row = {}
    row['customerID']       = d.get('customerID')
    row['Churn']            = d.get('Churn')
    cust = d.get('customer', {})
    row['gender']           = cust.get('gender')
    row['SeniorCitizen']    = cust.get('SeniorCitizen')
    row['Partner']          = cust.get('Partner')
    row['Dependents']       = cust.get('Dependents')
    row['tenure']           = cust.get('tenure')
    phone = d.get('phone', {})
    row['PhoneService']     = phone.get('PhoneService')
    row['MultipleLines']    = phone.get('MultipleLines')
    internet = d.get('internet', {})
    row['InternetService']  = internet.get('InternetService')
    row['OnlineSecurity']   = internet.get('OnlineSecurity')
    row['OnlineBackup']     = internet.get('OnlineBackup')
    row['DeviceProtection'] = internet.get('DeviceProtection')
    row['TechSupport']      = internet.get('TechSupport')
    row['StreamingTV']      = internet.get('StreamingTV')
    row['StreamingMovies']  = internet.get('StreamingMovies')
    account = d.get('account', {})
    row['Contract']         = account.get('Contract')
    row['PaperlessBilling'] = account.get('PaperlessBilling')
    row['PaymentMethod']    = account.get('PaymentMethod')
    charges = account.get('Charges', {})
    row['Charges_Monthly']  = charges.get('Monthly')
    row['Charges_Total']    = charges.get('Total')
    records.append(row)

df_raw = pd.DataFrame(records)

# ── Limpeza (Parte 1) ─────────────────────────────────
df = df_raw[df_raw['Churn'].str.strip() != ''].copy()
df['Charges_Total'] = pd.to_numeric(df['Charges_Total'], errors='coerce')
mask = df['Charges_Total'].isna()
df.loc[mask, 'Charges_Total'] = df.loc[mask,'Charges_Monthly'] * df.loc[mask,'tenure']
df['Contas_Diarias'] = (df['Charges_Monthly'] / 30).round(4)
df['Churn_num'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"✅ Dataset limpo: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head(3)


---
# 🛠️ Seção 2 — Pré-Processamento para Machine Learning

## 2.1 Remoção de Colunas Irrelevantes

In [ ]:
# customerID é apenas um identificador único — não tem poder preditivo
# Churn (string) é substituído por Churn_num (0/1)
cols_remover = ['customerID', 'Churn']

df_ml = df.drop(columns=cols_remover).copy()

print(f"Colunas removidas: {cols_remover}")
print(f"Colunas restantes ({df_ml.shape[1]}): {list(df_ml.columns)}")


## 2.2 Encoding — Variáveis Categóricas

In [ ]:
# Identificar colunas categóricas que ainda precisam de encoding
cat_cols = df_ml.select_dtypes(include='object').columns.tolist()
print(f"Colunas categóricas para encoding: {cat_cols}")
print()
for col in cat_cols:
    print(f"  {col}: {df_ml[col].unique()}")


In [ ]:
# One-Hot Encoding para variáveis com mais de 2 categorias
# (Contract, PaymentMethod, InternetService, gender, MultipleLines)
df_encoded = pd.get_dummies(df_ml, columns=cat_cols, drop_first=False)

# Converter colunas booleanas geradas para int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"✅ Encoding aplicado.")
print(f"   Colunas antes : {df_ml.shape[1]}")
print(f"   Colunas depois: {df_encoded.shape[1]}")
print(f"\nNovas colunas geradas pelo OHE:")
print([c for c in df_encoded.columns if c not in df_ml.columns])


## 2.3 Verificação da Proporção de Evasão

In [ ]:
churn_counts = df_encoded['Churn_num'].value_counts()
churn_pct    = df_encoded['Churn_num'].value_counts(normalize=True) * 100

print("Distribuição da variável alvo (Churn_num):")
print(f"  Não Evadiu (0): {churn_counts[0]:,}  ({churn_pct[0]:.1f}%)")
print(f"  Evadiu     (1): {churn_counts[1]:,}  ({churn_pct[1]:.1f}%)")
print(f"  Razão desbalanceamento: {churn_counts[0]/churn_counts[1]:.1f}:1")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Não Evadiu (0)', 'Evadiu (1)'], churn_counts.values,
       color=['#2ecc71','#e74c3c'], edgecolor='white', width=0.5)
ax.set_title('Proporção das Classes — Churn', fontweight='bold')
ax.set_ylabel('Quantidade')
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 30, f'{v:,}\n({churn_pct.iloc[i]:.1f}%)',
            ha='center', fontweight='bold')
ax.set_ylim(0, churn_counts.max() * 1.2)
plt.tight_layout()
plt.show()

print("\n⚠️  Há desbalanceamento: ~73.5% vs ~26.5%. Aplicaremos SMOTE para corrigir.")


## 2.4 Separação Features / Target

In [ ]:
X = df_encoded.drop(columns=['Churn_num'])
y = df_encoded['Churn_num']

print(f"Features (X): {X.shape}")
print(f"Target   (y): {y.shape}")
print(f"\nColunas de X:")
print(list(X.columns))


## 2.5 Balanceamento com SMOTE

In [ ]:
# SMOTE gera exemplos sintéticos da classe minoritária (Churn=1)
# para equilibrar as classes no conjunto de TREINO

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Antes do SMOTE:")
print(f"  Treino — Classe 0: {(y_train_raw==0).sum()}  |  Classe 1: {(y_train_raw==1).sum()}")

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_raw, y_train_raw)

print(f"\nApós SMOTE:")
print(f"  Treino — Classe 0: {(y_train_bal==0).sum()}  |  Classe 1: {(y_train_bal==1).sum()}")
print(f"  Total treino balanceado: {len(X_train_bal)}")
print(f"\n✅ Classes equilibradas para o treinamento.")


## 2.6 Normalização com StandardScaler

In [ ]:
# Necessário para: Regressão Logística (sensível à escala)
# Não necessário para: Random Forest e Gradient Boosting (baseados em árvore)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_bal)   # fit apenas no treino
X_test_scaled  = scaler.transform(X_test)            # aplica a mesma escala no teste

print("✅ Normalização aplicada (StandardScaler).")
print(f"   Média dos dados de treino (esperado ~0): {X_train_scaled.mean():.4f}")
print(f"   Desvio padrão dos dados de treino (esperado ~1): {X_train_scaled.std():.4f}")


---
# 🔗 Seção 3 — Correlação e Seleção de Variáveis

## 3.1 Matriz de Correlação

In [ ]:
# Correlação das variáveis numéricas originais com o Churn
num_cols_corr = ['SeniorCitizen','tenure','Charges_Monthly','Charges_Total',
                 'Contas_Diarias','Churn_num']

corr = df[num_cols_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 11},
            vmin=-1, vmax=1)
ax.set_title('Matriz de Correlação — Variáveis Numéricas', fontweight='bold')
plt.tight_layout()
plt.show()


## 3.2 Correlação de Todas as Features com Churn

In [ ]:
corr_target = df_encoded.corr()['Churn_num'].drop('Churn_num')
corr_target_sorted = corr_target.abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(11, 7))
cores = ['#e74c3c' if corr_target[i] > 0 else '#2ecc71'
         for i in corr_target_sorted.index]
bars = ax.barh(corr_target_sorted.index[::-1],
               corr_target[corr_target_sorted.index[::-1]],
               color=cores[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Top 20 Variáveis — Correlação com Churn', fontweight='bold')
ax.set_xlabel('Coeficiente de Correlação (Pearson)')
plt.tight_layout()
plt.show()

print("Top 10 variáveis mais correlacionadas com Churn:")
for var, val in corr_target.abs().sort_values(ascending=False).head(10).items():
    sinal = "+" if corr_target[var] > 0 else "-"
    print(f"  {sinal}{val:.4f}  →  {var}")


## 3.3 Análises Direcionadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot: Tenure × Churn
bp1 = axes[0].boxplot(
    [df[df['Churn']=='No']['tenure'], df[df['Churn']=='Yes']['tenure']],
    labels=['Não Evadiu', 'Evadiu'], patch_artist=True,
    medianprops=dict(color='black', linewidth=2.5)
)
bp1['boxes'][0].set_facecolor('#2ecc71')
bp1['boxes'][1].set_facecolor('#e74c3c')
axes[0].set_title('Tempo de Contrato × Evasão', fontweight='bold')
axes[0].set_ylabel('Meses de Contrato (Tenure)')

# Boxplot: Total Gasto × Churn
bp2 = axes[1].boxplot(
    [df[df['Churn']=='No']['Charges_Total'], df[df['Churn']=='Yes']['Charges_Total']],
    labels=['Não Evadiu', 'Evadiu'], patch_artist=True,
    medianprops=dict(color='black', linewidth=2.5)
)
bp2['boxes'][0].set_facecolor('#3498db')
bp2['boxes'][1].set_facecolor('#e74c3c')
axes[1].set_title('Total Gasto × Evasão', fontweight='bold')
axes[1].set_ylabel('Cobrança Total (R$)')

plt.suptitle('Análises Direcionadas — Variáveis Chave vs Churn',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Tenure   mediano — Não Evadiu: {df[df['Churn']=='No']['tenure'].median():.0f} meses")
print(f"Tenure   mediano — Evadiu    : {df[df['Churn']=='Yes']['tenure'].median():.0f} meses")
print(f"Total $ mediano  — Não Evadiu: R$ {df[df['Churn']=='No']['Charges_Total'].median():.2f}")
print(f"Total $ mediano  — Evadiu    : R$ {df[df['Churn']=='Yes']['Charges_Total'].median():.2f}")


In [ ]:
# Scatter: Tenure × Charges_Monthly colorido por Churn
fig, ax = plt.subplots(figsize=(10, 6))
cores_scatter = df['Churn_num'].map({0: '#2ecc71', 1: '#e74c3c'})
ax.scatter(df['tenure'], df['Charges_Monthly'],
           c=cores_scatter, alpha=0.35, s=18, edgecolors='none')
ax.set_xlabel('Meses de Contrato (Tenure)')
ax.set_ylabel('Cobrança Mensal (R$)')
ax.set_title('Dispersão: Tenure × Cobrança Mensal por Status de Churn', fontweight='bold')
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=9, label='Não Evadiu'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=9, label='Evadiu')
]
ax.legend(handles=legend_elements, fontsize=11)
plt.tight_layout()
plt.show()


---
# 🤖 Seção 4 — Modelagem Preditiva

## 4.1 Estratégia de Modelagem

Treinamos **4 modelos** com abordagens diferentes:

| Modelo | Requer Normalização | Justificativa |
|---|---|---|
| **Regressão Logística** | ✅ Sim | Modelo linear baseline, interpretável, sensível à escala |
| **Árvore de Decisão** | ❌ Não | Simples, interpretável visualmente, bom ponto de comparação |
| **Random Forest** | ❌ Não | Ensemble robusto, fornece importância de variáveis, resistente a overfitting |
| **Gradient Boosting** | ❌ Não | Alta performance, aprende erros sequencialmente |

Usamos **dados balanceados (SMOTE)** para treino e dados **originais (não balanceados)** para teste,
simulando o cenário real onde novas previsões virão de dados desbalanceados.


## 4.2 Treinamento dos Modelos

In [ ]:
# ── Definição dos modelos ────────────────────────────────────────────
modelos = {
    'Regressão Logística': LogisticRegression(max_iter=1000, random_state=42, C=0.5),
    'Árvore de Decisão':   DecisionTreeClassifier(max_depth=7, min_samples_leaf=20, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10,
                                                  min_samples_leaf=10, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                                       learning_rate=0.05, random_state=42)
}

# ── Dados de treino por tipo ─────────────────────────────────────────
dados_treino = {
    'Regressão Logística': (X_train_scaled, y_train_bal),   # normalizado
    'Árvore de Decisão':   (X_train_bal,    y_train_bal),   # não normalizado
    'Random Forest':       (X_train_bal,    y_train_bal),
    'Gradient Boosting':   (X_train_bal,    y_train_bal),
}
dados_teste = {
    'Regressão Logística': X_test_scaled,
    'Árvore de Decisão':   X_test,
    'Random Forest':       X_test,
    'Gradient Boosting':   X_test,
}

# ── Treinamento ──────────────────────────────────────────────────────
resultados = {}
for nome, modelo in modelos.items():
    X_tr, y_tr = dados_treino[nome]
    modelo.fit(X_tr, y_tr)
    y_pred = modelo.predict(dados_teste[nome])
    y_prob = modelo.predict_proba(dados_teste[nome])[:, 1]

    resultados[nome] = {
        'modelo': modelo,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'acuracia':  accuracy_score(y_test, y_pred),
        'precisao':  precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob),
    }
    print(f"✅ {nome} treinado.")


## 4.3 Avaliação dos Modelos

In [ ]:
# ── Tabela comparativa de métricas ──────────────────────────────────
metricas_df = pd.DataFrame({
    nome: {
        'Acurácia':  f"{r['acuracia']:.4f}",
        'Precisão':  f"{r['precisao']:.4f}",
        'Recall':    f"{r['recall']:.4f}",
        'F1-Score':  f"{r['f1']:.4f}",
        'ROC-AUC':   f"{r['roc_auc']:.4f}",
    }
    for nome, r in resultados.items()
}).T

print("=" * 75)
print("COMPARATIVO DE MÉTRICAS — TODOS OS MODELOS")
print("=" * 75)
display(metricas_df)


In [ ]:
# ── Gráfico comparativo de métricas ─────────────────────────────────
metricas_plot = ['acuracia', 'precisao', 'recall', 'f1', 'roc_auc']
labels_plot   = ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'ROC-AUC']
nomes = list(resultados.keys())
x = np.arange(len(labels_plot))
w = 0.18
cores_mod = ['#3498db','#e67e22','#2ecc71','#9b59b6']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (nome, cor) in enumerate(zip(nomes, cores_mod)):
    vals = [resultados[nome][m] for m in metricas_plot]
    bars = ax.bar(x + i*w, vals, w, label=nome, color=cor, edgecolor='white', alpha=0.9)

ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(labels_plot, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Comparativo de Métricas — Modelos de Classificação', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

for bar_group in ax.containers:
    ax.bar_label(bar_group, fmt='%.2f', fontsize=7, padding=2, rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# ── Matrizes de Confusão ────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (nome, r) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['Não Evadiu', 'Evadiu'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome}\nF1={r["f1"]:.3f}  AUC={r["roc_auc"]:.3f}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Previsto'); ax.set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — Todos os Modelos',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Curvas ROC ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
cores_roc = ['#3498db','#e67e22','#2ecc71','#9b59b6']

for (nome, r), cor in zip(resultados.items(), cores_roc):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    ax.plot(fpr, tpr, label=f"{nome} (AUC={r['roc_auc']:.3f})",
            color=cor, linewidth=2.5)

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Aleatório (AUC=0.5)')
ax.set_xlabel('Taxa de Falso Positivo (FPR)')
ax.set_ylabel('Taxa de Verdadeiro Positivo (TPR / Recall)')
ax.set_title('Curvas ROC — Comparativo dos Modelos', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
plt.tight_layout()
plt.show()


## 4.4 Análise de Overfitting / Underfitting

In [ ]:
print("=" * 65)
print("ANÁLISE DE OVERFITTING — Treino vs Teste (F1-Score)")
print("=" * 65)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nome, r in resultados.items():
    modelo = r['modelo']
    X_tr = X_train_scaled if nome == 'Regressão Logística' else X_train_bal
    y_pred_treino = modelo.predict(X_tr)
    f1_treino = f1_score(y_train_bal, y_pred_treino)
    f1_teste  = r['f1']
    diff = f1_treino - f1_teste
    status = "⚠️  OVERFITTING" if diff > 0.08 else ("⚠️  UNDERFITTING" if f1_teste < 0.5 else "✅ Adequado")
    print(f"  {nome:<25} | Treino: {f1_treino:.4f} | Teste: {f1_teste:.4f} | Δ: {diff:+.4f}  {status}")


---
# 🔍 Seção 5 — Interpretação e Importância das Variáveis

## 5.1 Regressão Logística — Coeficientes

In [ ]:
lr = resultados['Regressão Logística']['modelo']
coef_df = pd.DataFrame({
    'Variável': X.columns,
    'Coeficiente': lr.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=False).head(20)

fig, ax = plt.subplots(figsize=(11, 7))
cores_coef = ['#e74c3c' if v > 0 else '#2ecc71' for v in coef_df['Coeficiente'][::-1]]
ax.barh(coef_df['Variável'][::-1], coef_df['Coeficiente'][::-1],
        color=cores_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Regressão Logística — Top 20 Coeficientes\n(vermelho = aumenta churn  |  verde = reduz churn)',
             fontweight='bold')
ax.set_xlabel('Coeficiente')
plt.tight_layout()
plt.show()


## 5.2 Random Forest — Importância das Variáveis

In [ ]:
rf = resultados['Random Forest']['modelo']
imp_df = pd.DataFrame({
    'Variável': X.columns,
    'Importância': rf.feature_importances_
}).sort_values('Importância', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(11, 7))
palette = sns.color_palette('YlOrRd', len(imp_df))[::-1]
ax.barh(imp_df['Variável'][::-1], imp_df['Importância'][::-1],
        color=palette, edgecolor='white')
ax.set_title('Random Forest — Top 20 Variáveis Mais Importantes',
             fontweight='bold')
ax.set_xlabel('Importância (Gini Impurity)')
plt.tight_layout()
plt.show()

print("Top 10 variáveis (Random Forest):")
for _, row in imp_df.head(10).iterrows():
    print(f"  {row['Importância']:.4f}  →  {row['Variável']}")


## 5.3 Gradient Boosting — Importância das Variáveis

In [ ]:
gb = resultados['Gradient Boosting']['modelo']
imp_gb = pd.DataFrame({
    'Variável': X.columns,
    'Importância': gb.feature_importances_
}).sort_values('Importância', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(11, 7))
palette_gb = sns.color_palette('Blues_r', len(imp_gb))
ax.barh(imp_gb['Variável'][::-1], imp_gb['Importância'][::-1],
        color=palette_gb, edgecolor='white')
ax.set_title('Gradient Boosting — Top 20 Variáveis Mais Importantes',
             fontweight='bold')
ax.set_xlabel('Importância')
plt.tight_layout()
plt.show()


## 5.4 Consenso — Variáveis Mais Importantes entre os Modelos

In [ ]:
# Normalizar importâncias para comparação justa entre modelos
def normalizar(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn) if mx != mn else series

imp_lr = pd.Series(np.abs(lr.coef_[0]), index=X.columns)
imp_rf = pd.Series(rf.feature_importances_, index=X.columns)
imp_gb = pd.Series(gb.feature_importances_, index=X.columns)

consenso = pd.DataFrame({
    'Log. Reg.':  normalizar(imp_lr),
    'Rand. Forest': normalizar(imp_rf),
    'Grad. Boost':  normalizar(imp_gb),
})
consenso['Media'] = consenso.mean(axis=1)
top15_consenso = consenso.sort_values('Media', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
top15_consenso['Media'][::-1].plot(kind='barh', ax=ax,
                                    color='#8e44ad', edgecolor='white')
ax.set_title('Consenso de Importância — Top 15 Variáveis (média entre 3 modelos)',
             fontweight='bold')
ax.set_xlabel('Importância Normalizada Média')
plt.tight_layout()
plt.show()


---
# 📄 Seção 6 — Relatório Final e Conclusão Estratégica

In [ ]:
print("=" * 70)
print("     RELATÓRIO EXECUTIVO — MODELAGEM PREDITIVA DE CHURN")
print("     TelecomX Data Science Team")
print("=" * 70)

melhor = max(resultados, key=lambda n: resultados[n]['f1'])
print(f"\n🏆 MELHOR MODELO: {melhor}")
print(f"   F1-Score : {resultados[melhor]['f1']:.4f}")
print(f"   ROC-AUC  : {resultados[melhor]['roc_auc']:.4f}")
print(f"   Recall   : {resultados[melhor]['recall']:.4f}")
print(f"   Precisão : {resultados[melhor]['precisao']:.4f}")

print(f"\n📊 RANKING COMPLETO (por F1-Score):")
ranking = sorted(resultados.items(), key=lambda x: x[1]['f1'], reverse=True)
for i, (nome, r) in enumerate(ranking, 1):
    print(f"   {i}. {nome:<25} F1={r['f1']:.4f}  AUC={r['roc_auc']:.4f}  Recall={r['recall']:.4f}")

print(f"\n🔑 TOP 5 FATORES DE RISCO IDENTIFICADOS (consenso entre modelos):")
top5 = top15_consenso.head(5)
for i, (var, row) in enumerate(top5.iterrows(), 1):
    print(f"   {i}. {var:<45} (score: {row['Media']:.4f})")
print("=" * 70)


## 6.1 Introdução

Este relatório documenta a segunda fase do projeto TelecomX, focada na **construção e avaliação de modelos preditivos de Machine Learning** para identificar clientes com alto risco de cancelamento (Churn).

O objetivo é transformar os padrões identificados na análise exploratória em um sistema preditivo robusto, capaz de antecipar a evasão antes que ela ocorra, permitindo ações preventivas de retenção.

---

## 6.2 Pipeline de Pré-Processamento

### Dados de entrada
- **7.043 registros** limpos provenientes do ETL da Parte 1
- **21 variáveis** após remoção de identificadores

### Etapas aplicadas

| Etapa | Técnica | Justificativa |
|---|---|---|
| Remoção de colunas | Drop `customerID` | Sem valor preditivo |
| Encoding | One-Hot Encoding | Compatibilidade com algoritmos ML |
| Balanceamento | SMOTE (oversampling) | Corrige desbalanceamento 73%/27% |
| Normalização | StandardScaler | Aplicado apenas para Regressão Logística |
| Divisão | 70% treino / 30% teste | Avaliação em dados não vistos |

---

## 6.3 Modelos e Resultados

### Justificativa da escolha dos modelos

- **Regressão Logística:** baseline interpretável, coeficientes mostram direção do impacto de cada variável. Requer normalização por ser sensível à escala.
- **Árvore de Decisão:** modelo simples e visual, bom para identificar regras de decisão claras.
- **Random Forest:** ensemble que reduz overfitting via média de múltiplas árvores. Robusto e preciso, fornece importância de variáveis via Gini.
- **Gradient Boosting:** aprende sequencialmente os erros dos modelos anteriores, geralmente o de maior performance.

### Análise de Overfitting
Todos os modelos foram verificados comparando F1 no treino vs teste. O Random Forest e Gradient Boosting apresentaram generalização adequada com os hiperparâmetros escolhidos (max_depth limitado, min_samples_leaf ajustado).

---

## 6.4 Principais Fatores de Churn Identificados

Com base no **consenso entre os três modelos** com importância de variáveis:

### 🔴 Fatores que AUMENTAM o risco de Churn
1. **Contrato mês a mês** — maior fator individual de risco
2. **Serviço de fibra óptica** — alta cobrança com possível insatisfação
3. **Pagamento por cheque eletrônico** — baixo engajamento
4. **Cobrança mensal elevada** — custo percebido como excessivo
5. **Pouco tempo de contrato (tenure baixo)** — primeiros meses são críticos

### 🟢 Fatores que REDUZEM o risco de Churn
1. **Contrato de 2 anos** — comprometimento de longo prazo
2. **Tenure alto** — clientes fiéis têm menor propensão ao cancelamento
3. **Segurança online e suporte técnico** — serviços adicionais criam âncora
4. **Débito automático como pagamento** — maior engajamento
5. **Alto valor total gasto** — clientes com histórico longo raramente cancelam

---

## 6.5 Conclusão e Recomendações Estratégicas

| # | Ação | Impacto | Prioridade |
|---|---|---|---|
| 1 | **Usar o modelo preditivo** para identificar clientes em risco mensalmente e acionar time de retenção | Redução direta do churn | 🔴 Alta |
| 2 | **Incentivos para migração de contrato mensal → anual** nos primeiros 6 meses (desconto, bônus, upgrades) | Maior comprometimento | 🔴 Alta |
| 3 | **Onboarding proativo nos primeiros 12 meses** (calls, tutoriais, ofertas personalizadas) | Reduz churn precoce | 🔴 Alta |
| 4 | **Revisar custo-benefício da fibra óptica** — avaliar qualidade e percepção de valor | Reduz churn em fibra | 🟠 Média |
| 5 | **Pacotes de serviços adicionais** incluindo segurança online desde o plano básico | Aumenta âncora de retenção | 🟠 Média |
| 6 | **Estimular débito automático** como padrão, reduzindo fricção no pagamento | Reduz churn por método de pagamento | 🟡 Baixa |

---

### Próximos Passos para a Equipe de Data Science

- 🔧 **Tunar hiperparâmetros** do Gradient Boosting com GridSearchCV
- 📊 **Threshold personalizado** — ajustar o ponto de corte da probabilidade para maximizar Recall (priorizar detectar evasões)
- 🔁 **Pipeline de retraining** mensal com dados atualizados
- 📲 **API de inferência** para integrar o modelo ao CRM da TelecomX

---

*📊 Relatório produzido como parte do projeto Churn de Clientes — Fase 2: Modelagem*  
*🐍 Python · Scikit-learn · Pandas · Matplotlib · Seaborn · Imbalanced-learn*
